### 0️. Importar Librerias

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [2]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.preprocessing import (
    StandardScaler
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,

    confusion_matrix,
    classification_report
)

from sklearn.linear_model import (
    LinearRegression, 
    LogisticRegression, 
    Ridge, 
    Lasso, 
    ElasticNet)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


# Para que se muestren todas las columnas al inspeccionar los DataFrames
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

In [3]:
from framework import sp_carga_exploracion as sc
from framework import sp_limpieza_transformacion as sl
from framework import sp_eda as se
from framework import sp_abtest as sb
from framework import sp_modelado as sm

### 1️. Cargar DataFrame 

#### DataFrame 

In [5]:
ruta = (r"../data/processed/03_datos_eda.csv")
df_md = sc.leer_csv(ruta)

CSV CARGADO
Archivo: ../data/processed/03_datos_eda.csv
Filas:   1000
Columnas: 14


####  escuela copia del df_md

In [6]:
escuela = df_md.copy()

### 2. Análisis de Datos

#### clasificar_columnas()

In [7]:
sm.clasificar_columnas(escuela)

,columna,dtype,n_unicos,sugerencia,motivo
0,tiene_tutor_sn,str,2,col_control,Categórica con 2 valores únicos: rango razonable para agrupar
1,aprobado_sn,str,2,col_control,Categórica con 2 valores únicos: rango razonable para agrupar
2,nivel_dificultad,str,3,col_control,Categórica con 3 valores únicos: rango razonable para agrupar
3,horario_estudio_preferido,str,4,col_control,Categórica con 4 valores únicos: rango razonable para agrupar
4,estilo_aprendizaje,str,5,col_control,Categórica con 5 valores únicos: rango razonable para agrupar
5,rendimiento_academico,str,5,col_control,Categórica con 5 valores únicos: rango razonable para agrupar
6,nota_final,float64,364,metrica,Numérica con 364 valores únicos: variable continua medible
7,horas_sueno,float64,811,metrica,Numérica con 811 valores únicos: variable continua medible
8,tasa_asistencia,float64,893,metrica,Numérica con 893 valores únicos: variable continua medible
9,horas_estudio_semanal,float64,964,metrica,Numérica con 964 valores únicos: variable continua medible


In [8]:
escuela.columns.tolist()

['horas_estudio_semanal',
 'nota_anterior',
 'tasa_asistencia',
 'horas_sueno',
 'edad',
 'nivel_dificultad',
 'tiene_tutor_sn',
 'horario_estudio_preferido',
 'estilo_aprendizaje',
 'nota_final',
 'aprobado_ml',
 'aprobado_sn',
 'tiene_tutor_ml',
 'rendimiento_academico']

#### Regresión

##### target values Regresión

In [10]:
tv_reg = 'nota_final'

##### cols_excluir_reg

In [11]:
# Porque derivan directa o indirectamente de nota_final y son tv_clf.
cols_excluir_reg = ['aprobado_ml','aprobado_sn','rendimiento_academico','tiene_tutor_sn']

 ##### datasets especifico

In [15]:
escuela_reg = escuela.drop(columns=cols_excluir_reg,errors='ignore')
escuela_reg.sample(3)

,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,nivel_dificultad,horario_estudio_preferido,estilo_aprendizaje,nota_final,tiene_tutor_ml
537,14.186692,60.176043,50.481145,6.551155,24,facil,manana,lectura_escritura,71.5,1
974,1.000000,30.224020,20.000000,4.339117,23,medio,desconocido,visual,50.8,0
158,3.762473,76.214003,66.714663,6.439614,19,facil,tarde,desconocido,67.8,0


##### predictorias 

In [21]:
predictoras_reg = escuela_reg.drop(columns=[tv_reg]).columns.tolist()
predictoras_reg 

['horas_estudio_semanal',
 'nota_anterior',
 'tasa_asistencia',
 'horas_sueno',
 'edad',
 'nivel_dificultad',
 'horario_estudio_preferido',
 'estilo_aprendizaje',
 'tiene_tutor_ml']

#### Clasificación

##### target values Clasificacion

In [9]:
tv_clf = 'aprobado_ml'

In [12]:
# Porque derivan directa o indirectamente de aprobado y nota final y son tv_reg
cols_excluir_clf = ['nota_final','aprobado_sn','rendimiento_academico','tiene_tutor_sn']

In [17]:
escuela_clf = escuela.drop(columns=cols_excluir_clf,errors='ignore')
escuela_clf.sample(3)

,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,nivel_dificultad,horario_estudio_preferido,estilo_aprendizaje,aprobado_ml,tiene_tutor_ml
37,3.390271,68.132150,59.162324,7.007670,25,facil,tarde,visual,1,0
473,7.149195,64.031231,27.695047,7.629611,29,medio,noche,visual,0,1
94,17.451676,59.561381,67.041642,10.000000,18,facil,manana,kinestesico,1,1


In [22]:
predictoras_clf = escuela_clf.drop(columns=[tv_clf]).columns.tolist()
predictoras_clf 

['horas_estudio_semanal',
 'nota_anterior',
 'tasa_asistencia',
 'horas_sueno',
 'edad',
 'nivel_dificultad',
 'horario_estudio_preferido',
 'estilo_aprendizaje',
 'tiene_tutor_ml']

## A. REGRESION 

### 3. Preparación de Datos

In [25]:
escuela_reg = sm.preparacion_dataframe(escuela_reg)
escuela_reg.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   horas_estudio_semanal      1000 non-null   float64
 1   nota_anterior              1000 non-null   float64
 2   tasa_asistencia            1000 non-null   float64
 3   horas_sueno                1000 non-null   float64
 4   edad                       1000 non-null   int64  
 5   nivel_dificultad           1000 non-null   str    
 6   horario_estudio_preferido  1000 non-null   str    
 7   estilo_aprendizaje         1000 non-null   str    
 8   nota_final                 1000 non-null   float64
 9   tiene_tutor_ml             1000 non-null   int64  
dtypes: float64(5), int64(2), str(3)
memory usage: 98.3 KB


### 4. Separar Variables

#### Variables para modelado

In [32]:
escuela_reg.sample(3)

,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,nivel_dificultad,horario_estudio_preferido,estilo_aprendizaje,nota_final,tiene_tutor_ml
592,13.908130,90.967548,73.064949,8.321332,20,medio,manana,kinestesico,82.2,0
385,17.607625,70.583522,93.833996,10.000000,26,facil,manana,kinestesico,85.6,1
450,12.016778,70.432886,47.947545,5.452737,28,medio,noche,auditivo,70.1,0


In [26]:
tv_reg

'nota_final'

In [29]:
X_reg, y_reg = sm.separar_xy(
    escuela_reg,
    tv_reg
)

In [34]:
    print(X_reg.shape)
    print(y_reg.shape)

    display(X_reg.head())
    display(y_reg.head())

(1000, 9)
(1000,)


,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,nivel_dificultad,horario_estudio_preferido,estilo_aprendizaje,tiene_tutor_ml
0,8.957476,48.830601,86.640182,6.675694,25,facil,tarde,lectura_escritura,1
1,11.042524,80.825707,83.449655,4.616844,18,dificil,tarde,desconocido,0
2,4.510776,90.383694,74.623607,7.755246,25,facil,manana,lectura_escritura,0
3,6.647213,81.878257,82.849841,8.592826,23,facil,desconocido,visual,0
4,1.000000,66.254179,54.539935,6.671840,21,medio,desconocido,auditivo,0


0    84.4
1    72.0
2    80.0
3    78.2
4    66.0
Name: nota_final, dtype: float64

Conclusiones

Se separa la variable objetivo (`nota_final`) de las variables predictoras.

Resultado:

- Observaciones: 1.000
- Variables predictoras: 9
- Variable objetivo: nota_final

Los conjuntos quedan preparados para la fase de transformación de variables.

### 5. Transformación de Variables

#### nulos 

In [35]:
escuela_reg.isna().sum()

horas_estudio_semanal        0
nota_anterior                0
tasa_asistencia              0
horas_sueno                  0
edad                         0
nivel_dificultad             0
horario_estudio_preferido    0
estilo_aprendizaje           0
nota_final                   0
tiene_tutor_ml               0
dtype: int64

✓ No hay nulos.
✓ tiene_tutor_ml ya está codificada.
✓ No es necesario codificar variables binarias.
✓ Hay 3 variables categóricas que requieren One Hot Encoding.

#### variables_numericas / categoricas / binarias  

In [37]:
escuela_reg.dtypes

horas_estudio_semanal        float64
nota_anterior                float64
tasa_asistencia              float64
horas_sueno                  float64
edad                           int64
nivel_dificultad                 str
horario_estudio_preferido        str
estilo_aprendizaje               str
nota_final                   float64
tiene_tutor_ml                 int64
dtype: object

In [38]:
numericas_reg = [
    'horas_estudio_semanal',
    'nota_anterior',
    'tasa_asistencia',
    'horas_sueno',
    'edad',
]

categoricas_reg = [
    'nivel_dificultad',
    'horario_estudio_preferido',
    'estilo_aprendizaje'
]

binaria_ml = [
    'tiene_tutor_ml'
]

 #### tratar_numericas()

In [43]:
# Numéricas
X_reg = sm.tratar_numericas(
    X_reg,
    numericas_reg
)

#### tratar_categoricas()

In [40]:
# Categóricas
X_reg = sm.tratar_categoricas(
    X_reg,
    categoricas_reg
)



#### codificar_categoricas()

In [41]:
# One Hot Encoding
X_reg = sm.codificar_categoricas(
    X_reg,
    categoricas_reg
)

In [44]:
    print(X_reg.shape)

    display(X_reg.head())

(1000, 18)


,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,tiene_tutor_ml,nivel_dificultad_dificil,nivel_dificultad_facil,nivel_dificultad_medio,horario_estudio_preferido_desconocido,horario_estudio_preferido_manana,horario_estudio_preferido_noche,horario_estudio_preferido_tarde,estilo_aprendizaje_auditivo,estilo_aprendizaje_desconocido,estilo_aprendizaje_kinestesico,estilo_aprendizaje_lectura_escritura,estilo_aprendizaje_visual
0,8.957476,48.830601,86.640182,6.675694,25,1,0,1,0,0,0,0,1,0,0,0,1,0
1,11.042524,80.825707,83.449655,4.616844,18,0,1,0,0,0,0,0,1,0,1,0,0,0
2,4.510776,90.383694,74.623607,7.755246,25,0,0,1,0,0,1,0,0,0,0,0,1,0
3,6.647213,81.878257,82.849841,8.592826,23,0,0,1,0,1,0,0,0,0,0,0,0,1
4,1.000000,66.254179,54.539935,6.671840,21,0,0,0,1,1,0,0,0,1,0,0,0,0


In [45]:
X_reg.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 18 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   horas_estudio_semanal                  1000 non-null   float64
 1   nota_anterior                          1000 non-null   float64
 2   tasa_asistencia                        1000 non-null   float64
 3   horas_sueno                            1000 non-null   float64
 4   edad                                   1000 non-null   int64  
 5   tiene_tutor_ml                         1000 non-null   int64  
 6   nivel_dificultad_dificil               1000 non-null   int64  
 7   nivel_dificultad_facil                 1000 non-null   int64  
 8   nivel_dificultad_medio                 1000 non-null   int64  
 9   horario_estudio_preferido_desconocido  1000 non-null   int64  
 10  horario_estudio_preferido_manana       1000 non-null   int64  
 11  horario_estudio_

### 6. División Train/Test